# Construction of the Twenty-Two Firm Characteristics

Builds the monthly panel and the full set of characteristics of Drobetz and Otto
(2021), whose definitions follow Drobetz, Haller, Jasperneite and Otto (2019).

Each characteristic is constructed in a numbered section reproducing the definition
given in the source study, followed by its implementation and a comparison against
the reported cross-sectional mean.

## Panel structure

Row $(i,t)$ holds characteristics observable at the end of month $t$ and, as
dependent variable, the excess return realised over month $t+1$. Market data enter
without delay; accounting data are lagged four months from the fiscal period end.

## Sources of ambiguity

Three points are not fully determined by the published definitions and are resolved
here on the evidence, with the reasoning recorded in the relevant section:

- the market benchmark for beta, unspecified in the 2021 study;
- the scaling of turnover, described only as "average monthly turnover volume";
- whether return volatility is reported in weekly or annualised units.

## 1b. Definitions checked against the source studies

Each definition was compared against Table 1 of Drobetz and Otto (2021) and the
corresponding table of Drobetz, Haller, Jasperneite and Otto (2019). The month index
in those tables is relative to the month whose return is being predicted, so that
month $-1$ is the current panel row, month $-2$ the preceding one, and so forth. Under
that convention:

- `ret_1` is the excess return of the current row; `ret_2_12` cumulates the eleven
  months from $t-11$ to $t-1$; `ret_12_36` the twenty-five months from $t-35$ to $t-11$;
- `turnover`, `dy`, `fcdispersion` and `vola` average or cumulate the twelve months
  from $t-11$ to $t$; `beta` the thirty-six months from $t-35$ to $t$;
- the issuance measures compare the level at month $-1$ with the level at month $-36$
  or $-12$, that is thirty-five and eleven rows earlier respectively.

**Three quantities are inferred rather than stated**, each resolved on the reported
magnitudes:

*Units of market value.* The reported mean of `size` is 15.60 for the
large-capitalisation sample of the 2019 study and 13.22 for the broader 2021 one.
Exponentiating gives €5.96 billion and €551 million if market value is expressed in
thousands, against €5.96 trillion and €551 billion if in millions; only the former is
possible. Market value is therefore in thousands in the source, against millions
here, a constant difference of $\log 1000$ that the rank transformation removes.

*Units of volatility.* The reported means are 0.30 and 0.32. As a weekly standard
deviation this would be implausible; annualised, it corresponds to a weekly standard
deviation near 4.3%, which is not.

*Scaling of turnover.* Described only as "average monthly turnover volume", with
reported means of 0.13 for the large-capitalisation sample and 0.02 for the broader
one. Raw volume would be of the order of millions of shares. Both magnitudes, and
their ordering, are consistent with volume scaled by shares outstanding.

**One reported statistic cannot be reconciled** with its stated definition.
`investment` is defined as capital expenditures over net sales, and is reported with
a mean of 1.01 in 2019 and 1.02 in 2021 — implying capital expenditure equal to
turnover for the average firm. The stated definition is implemented here, giving
values an order of magnitude smaller and economically plausible; the discrepancy is
noted rather than accommodated.

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import numpy as np
import pandas as pd

DATA_DIR = "/content/drive/MyDrive/Thesis/data"

monthly = pd.read_parquet(os.path.join(DATA_DIR, "dedup_monthly.parquet"))
weekly  = pd.read_parquet(os.path.join(DATA_DIR, "dedup_weekly.parquet"))
annual  = pd.read_parquet(os.path.join(DATA_DIR, "raw_annual.parquet"))
fye     = pd.read_parquet(os.path.join(DATA_DIR, "fiscal_year_end.parquet"))
rf_raw  = pd.read_parquet(os.path.join(DATA_DIR, "raw_riskfree.parquet"))
idx_raw = pd.read_parquet(os.path.join(DATA_DIR, "raw_index.parquet"))
master  = pd.read_csv(os.path.join(DATA_DIR, "master_universe.csv"), dtype=str)

for nm, d in [("monthly", monthly), ("weekly", weekly), ("annual", annual)]:
    print(f"{nm:8s} {len(d):>12,} rows | {d['symbol'].nunique():>6,} securities")
print(f"\nindices: {sorted(idx_raw['symbol'].unique())}")
print(f"risk-free: {len(rf_raw)} observations")

monthly     6,724,638 rows |  6,156 securities
weekly      3,166,425 rows |  5,660 securities
annual      1,028,439 rows |  5,200 securities

indices: ['TOTMKBD', 'TOTMKES', 'TOTMKEU', 'TOTMKFR', 'TOTMKIT']
risk-free: 360 observations


## 2. Functions

In [3]:
import numpy as np
import pandas as pd


def build_skeleton(monthly_long):
    """Wide panel on a continuous monthly calendar, per security."""
    w = monthly_long.pivot_table(index=["symbol", "date"], columns="datatype",
                                 values="value", aggfunc="first")
    w.columns.name = None
    rng = w.reset_index().groupby("symbol")["date"].agg(["min", "max"])
    idx = []
    for sym, (d0, d1) in rng.iterrows():
        dates = pd.date_range(d0, d1, freq="MS")
        idx.append(pd.MultiIndex.from_product([[sym], dates], names=["symbol", "date"]))
    full = idx[0].append(idx[1:]) if len(idx) > 1 else idx[0]
    return w.reindex(full).sort_index()


def add_returns(p, rf, apply_reversal_filter=True):
    """
    Simple and excess returns from the total return index.

    A total return index of zero or below is meaningless and is treated as
    missing; it otherwise produces infinite returns for securities quoted at
    rounding-level prices.

    The reversal filter of Ince and Porter (2006), designed for Datastream
    equity data, removes spurious return pairs: where a return above 300% is
    reversed within one month, both observations are set to missing. Such pairs
    arise from rounding in the price of very low-priced securities rather than
    from genuine price movements.
    """
    p = p.copy()
    ri = p["RI"].where(p["RI"] > 0)
    p["ret"] = ri.groupby(level="symbol").pct_change()
    p["ret"] = p["ret"].replace([np.inf, -np.inf], np.nan)

    if apply_reversal_filter:
        r = p["ret"]
        r_prev = r.groupby(level="symbol").shift(1)
        pair = (1 + r) * (1 + r_prev) - 1
        bad = ((r > 3.0) | (r_prev > 3.0)) & (pair < 0.5)
        bad = bad.fillna(False)
        n = int(bad.sum())
        p.loc[bad, "ret"] = np.nan
        # the paired observation is discarded as well
        bad_next = bad.groupby(level="symbol").shift(-1).fillna(False).astype(bool)
        p.loc[bad_next, "ret"] = np.nan
        p["_rev_filtered"] = bad | bad_next

    p = p.join(rf.rename("rf"), on="date")
    p["rf"] = p["rf"] / 100.0 / 12.0          # annual percent -> monthly decimal
    p["retx"] = p["ret"] - p["rf"]
    return p


def _cum_window(s, start_lag, n):
    """Cumulative return over the n months ending `start_lag` months back."""
    lr = np.log1p(s.clip(lower=-0.99))
    roll = (lr.groupby(level="symbol")
              .apply(lambda x: x.shift(start_lag).rolling(n, min_periods=n).sum())
              .droplevel(0))
    return np.expm1(roll.reindex(s.index))


def add_market_characteristics(p):
    """Characteristics computable from monthly market and I/B/E/S data."""
    p = p.copy()
    gb = lambda c: p.groupby(level="symbol")[c]

    # size
    p["size"] = np.log(p["MV"].where(p["MV"] > 0))

    # momentum and reversal
    p["ret_1"] = p["retx"]
    p["ret_2_12"] = _cum_window(p["retx"], 1, 11)
    p["ret_12_36"] = _cum_window(p["retx"], 11, 25)

    # Share issuance, adjusted for capital events.
    # Defined as growth "from month -36 to month -1". Indexing months relative
    # to the month whose return is being predicted, month -1 is the current
    # panel row and month -36 is thirty-five rows earlier.
    adj = (p["NOSH"] / p["AF"]).replace([np.inf, -np.inf], np.nan)
    adj = adj.where(adj > 0)
    p["_adj_sh"] = adj
    la = np.log(adj)
    p["issues_1_12"] = la - gb("_adj_sh").shift(11).pipe(np.log)
    p["issues_1_36"] = la - gb("_adj_sh").shift(35).pipe(np.log)

    # turnover: shares traded over shares outstanding, averaged over 12 months
    to = (p["VO"] / p["NOSH"]).replace([np.inf, -np.inf], np.nan)
    p["_to"] = to
    p["turnover"] = (p.groupby(level="symbol")["_to"]
                       .apply(lambda x: x.rolling(12, min_periods=6).mean())
                       .droplevel(0).reindex(p.index))

    # Dividend yield. The Datastream dividend per share is already a trailing
    # twelve-month figure reported monthly: verified against the DY datatype,
    # DPS/P reproduces it exactly, whereas summing DPS over a twelve-month
    # window overstates it by a factor of about twelve.
    p["dy"] = p["DPS"] / p["P"].where(p["P"] > 0)

    # forecast dispersion
    sd = p["EPS1SD"].where(p["EPS1SD"] > 0)
    mn = p["EPS1MN"].abs().replace(0, np.nan)
    p["_fc"] = np.log(sd) - np.log(mn)
    p["fcdispersion"] = (p.groupby(level="symbol")["_fc"]
                           .apply(lambda x: x.rolling(12, min_periods=3).mean())
                           .droplevel(0).reindex(p.index))

    return p.drop(columns=[c for c in p.columns if c.startswith("_")])


def add_target(p):
    """Dependent variable: excess return realised in the following month."""
    p = p.copy()
    p["target"] = p.groupby(level="symbol")["retx"].shift(-1)
    return p


def winsorise_returns(p, lower=0.01, upper=0.99, cols=("ret", "retx")):
    """
    Winsorise returns cross-sectionally, month by month.

    Applied within each cross-section rather than over the pooled sample, so
    that no information from future periods enters the transformation.
    """
    p = p.copy()
    for c in cols:
        lo = p.groupby(level="date")[c].transform(lambda s: s.quantile(lower))
        hi = p.groupby(level="date")[c].transform(lambda s: s.quantile(upper))
        p[c] = p[c].clip(lo, hi)
    return p


def apply_price_screen(p, threshold=0.25):
    """
    Price screen of Ince and Porter (2006).

    Datastream rounds prices to the nearest cent and the return index to the
    nearest tenth. At low price levels the rounding itself generates large
    spurious returns, which no filter based on the return series alone can
    fully remove. Observations whose price at the end of the preceding month
    falls below the threshold are therefore set to missing.

    Ince and Porter use one dollar and note that thresholds as low as 0.10 or
    0.25 perform almost as well; the lower value is adopted here, given the
    prevalence of low-priced securities on the smaller European exchanges.
    """
    p = p.copy()
    prev_price = p.groupby(level="symbol")["P"].shift(1)
    below = (prev_price < threshold) & prev_price.notna()
    p.loc[below, ["ret", "retx"]] = np.nan
    p["_price_screened"] = below
    return p


In [4]:
"""Construction of the twenty-two firm characteristics of Drobetz and Otto (2021)."""
import numpy as np
import pandas as pd

# ---------------------------------------------------------------- benchmark ---

def build_composite_index(index_long, monthly_panel, master, codes):
    """
    Value-weighted euro-area benchmark from the national total market indices.

    Country weights are the aggregate market capitalisation of the sample
    securities in that country, taken at the *end of the preceding month* so
    that no contemporaneous information enters the weights.
    """
    idx = (index_long[index_long["symbol"].isin(codes.values())]
           .pivot_table(index="date", columns="symbol", values="value", aggfunc="first")
           .sort_index())
    ret = idx.pct_change()

    mv = monthly_panel.reset_index()[["symbol", "date", "MV"]].dropna()
    mv = mv.merge(master[["Symbol", "country"]], left_on="symbol", right_on="Symbol")
    agg = mv.groupby(["date", "country"])["MV"].sum().unstack("country")
    agg = agg.shift(1)                                   # weights known ex ante
    w = agg.div(agg.sum(axis=1), axis=0)

    w = w.rename(columns={c: codes[c] for c in codes if c in w.columns})
    w = w.reindex(ret.index, method="ffill")

    cols = [c for c in ret.columns if c in w.columns]
    wn = w[cols].div(w[cols].sum(axis=1), axis=0)
    mkt = (ret[cols] * wn).sum(axis=1, min_count=1)
    mkt.name = "mkt_ret"
    return mkt, wn


# ------------------------------------------------------- weekly moments ---

def beta_and_vola(weekly_long, mkt_ret, beta_weeks=156, vola_weeks=52,
                  min_beta=104, min_vola=35):
    """
    Market beta and return volatility from weekly returns.

    Beta is estimated over the thirty-six months to month t-1 and volatility
    over the twelve months to t-1, following Drobetz and Otto (2021). Rolling
    covariances are used rather than repeated regressions.
    """
    ri = weekly_long[(weekly_long["datatype"] == "RI") & (weekly_long["value"] > 0)]
    w = (ri.pivot_table(index="date", columns="symbol", values="value", aggfunc="first")
           .sort_index())
    r = w.pct_change()
    r = r.replace([np.inf, -np.inf], np.nan)

    m = mkt_ret.reindex(r.index)
    mm = m.rolling(beta_weeks, min_periods=min_beta).mean()
    mv_ = m.rolling(beta_weeks, min_periods=min_beta).var()

    rm = r.rolling(beta_weeks, min_periods=min_beta).mean()
    prod = r.mul(m, axis=0)
    pm = prod.rolling(beta_weeks, min_periods=min_beta).mean()
    cov = pm.sub(rm.mul(mm, axis=0))
    beta = cov.div(mv_, axis=0)

    vola = r.rolling(vola_weeks, min_periods=min_vola).std()

    def to_monthly(df, name):
        s = df.stack(future_stack=True).rename(name).reset_index()
        s.columns = ["date", "symbol", name]
        s["month"] = s["date"].values.astype("datetime64[M]")
        s = s.sort_values("date").groupby(["symbol", "month"]).last()
        return s[name]

    b = to_monthly(beta, "beta")
    v = to_monthly(vola, "vola")
    out = pd.concat([b, v], axis=1)
    out.index.names = ["symbol", "date"]
    return out


# ------------------------------------------------------------ accounting ---

def accounting_available(annual_long, fy_long, lag_months=4):
    """
    Wide annual accounting data with the date from which each observation may
    be used, being the fiscal period end plus the reporting lag.
    """
    a = annual_long.pivot_table(index=["symbol", "date"], columns="datatype",
                                values="value", aggfunc="first")
    a.columns.name = None
    a = a.reset_index().rename(columns={"date": "fy"})

    fy = fy_long[["symbol", "date", "value_date"]].rename(
        columns={"date": "fy", "value_date": "fye"})
    a = a.merge(fy, on=["symbol", "fy"], how="left")

    # fall back on a December year-end where the reported date is missing
    a["fye"] = a["fye"].fillna(pd.to_datetime(a["fy"].astype(str) + "-12-31"))
    # Worldscope reports in thousands of currency units, Datastream market value
    # in millions. Verified against the sample: the ratio of common equity to
    # market value has a median of 339, implying a book-to-market of 0.34.
    for c in [c for c in a.columns if c.startswith("WC")]:
        a[c] = a[c] / 1000.0

    a["fye"] = pd.to_datetime(a["fye"])
    a["available"] = a["fye"] + pd.DateOffset(months=lag_months)
    a["available"] = pd.to_datetime(
        a["available"].values.astype("datetime64[M]")).astype("datetime64[ns]")
    return a.sort_values(["symbol", "available"])


def add_accounting_characteristics(panel, acc):
    """
    Merge accounting data onto the monthly panel as of the date each fiscal
    year became available, then form the accounting-based characteristics.
    """
    acc = acc.copy().sort_values(["symbol", "fy"])
    g = acc.groupby("symbol")

    # Prior-year levels, for growth rates and averages. Where a fiscal year is
    # missing from the record, the preceding available year is not a valid
    # comparison and the lagged level is discarded, so that a two-year change
    # is not treated as an annual one.
    consecutive = (acc["fy"] - g["fy"].shift(1)) == 1
    for c in ["WC02999", "WC02201", "WC02003", "WC03101", "WC03051", "WC03063"]:
        acc[f"{c}_lag"] = g[c].shift(1).where(consecutive)

    avg_ta = (acc["WC02999"] + acc["WC02999_lag"]) / 2
    avg_ta = avg_ta.where(avg_ta > 0)

    acc["operatingprofitability"] = acc["WC18191"] / avg_ta
    acc["grossprofitability"] = (acc["WC01001"] - acc["WC01051"]) / avg_ta
    acc["roa"] = acc["WC01551"] / avg_ta
    acc["totalassetgrowth"] = np.log(acc["WC02999"].where(acc["WC02999"] > 0)) - \
                              np.log(acc["WC02999_lag"].where(acc["WC02999_lag"] > 0))
    acc["investment"] = acc["WC04601"] / acc["WC01001"].where(acc["WC01001"] > 0)

    # Sloan (1996) working capital accruals
    d_ca = acc["WC02201"] - acc["WC02201_lag"]
    d_cash = acc["WC02003"] - acc["WC02003_lag"]
    d_cl = acc["WC03101"] - acc["WC03101_lag"]
    d_std = acc["WC03051"] - acc["WC03051_lag"]
    d_tp = acc["WC03063"] - acc["WC03063_lag"]
    acc["accrualschange"] = ((d_ca - d_cash) - (d_cl - d_std - d_tp) - acc["WC01151"]) / avg_ta

    cols = ["symbol", "available", "fy", "fye",
            "WC03501", "WC01751", "WC01001", "WC01151", "WC03051", "WC03251",
            "operatingprofitability", "grossprofitability", "roa",
            "totalassetgrowth", "investment", "accrualschange"]
    acc = acc[cols].dropna(subset=["available"])

    p = panel.reset_index().sort_values("date")
    p["date"] = pd.to_datetime(p["date"]).astype("datetime64[ns]")
    merged = pd.merge_asof(p, acc.sort_values("available"),
                           left_on="date", right_on="available",
                           by="symbol", direction="backward")
    merged = merged.set_index(["symbol", "date"]).sort_index()

    mv = merged["MV"].where(merged["MV"] > 0)
    merged["bm"] = np.log(merged["WC03501"].where(merged["WC03501"] > 0)) - np.log(mv)
    merged["earningstoprice"] = merged["WC01751"] / mv
    merged["salestoprice"] = merged["WC01001"] / mv
    merged["cftoprice"] = (merged["WC01751"] + merged["WC01151"]) / mv
    debt = merged[["WC03051", "WC03251"]].sum(axis=1, min_count=1)
    merged["debttoprice"] = debt / mv
    return merged


## 3. Panel skeleton, returns and data screens

Each security is placed on an uninterrupted monthly calendar. Excess returns are
computed from the total return index in excess of the three-month EURIBOR scaled to
a monthly horizon.

Three screens from Ince and Porter (2006), the standard reference on the cleaning of
Datastream equity data, are applied to the return series. Drobetz and Otto do not
discuss them, addressing outliers through winsorisation alone; the two treatments are
complementary rather than alternative, since winsorisation compresses extreme values
but does not distinguish spurious observations from genuine ones.

**Non-positive return index.** A total return index of zero yields an infinite return
and is set to missing.

**Reversal filter.** Where a return above 300% is reversed within one month, both
observations are discarded. Ince and Porter set both $R_t$ and $R_{t-1}$ to missing
when either exceeds 300% and $(1+R_t)(1+R_{t-1})-1$ falls below 50%.

**Price screen.** Datastream rounds prices to the nearest cent and the return index
to the nearest tenth, so that at low price levels the rounding itself generates large
spurious returns that no return-based filter can fully remove. Observations whose
prior-month price falls below a threshold are set to missing.

The screen on prices is the single most effective of the three: raising the threshold
from zero to twenty-five cents reduces the standard deviation of excess returns by a
factor of roughly fifty, at the cost of four per cent of observations.

In [5]:
rf = rf_raw.set_index("date")["value"]

panel = build_skeleton(panel_input := monthly)
panel = add_returns(panel, rf, apply_reversal_filter=True)

print(f"Panel: {len(panel):,} security-months, "
      f"{panel.index.get_level_values('symbol').nunique():,} securities")
if "_rev_filtered" in panel:
    print(f"Removed by reversal filter: {int(panel['_rev_filtered'].sum()):,}")

# price screen -- compare thresholds before applying
prev_price = panel.groupby(level="symbol")["P"].shift(1)
print("\nEffect of the price screen on excess returns:")
for thr in [0.00, 0.10, 0.25, 1.00]:
    r = panel["retx"].where((prev_price >= thr) | prev_price.isna())
    print(f"  threshold EUR {thr:4.2f}: n={r.notna().sum():>9,}  "
          f"std={r.std():>8.3f}  max={r.max():>9.1f}")

PRICE_SCREEN = 0.25
panel = apply_price_screen(panel, threshold=PRICE_SCREEN)
print(f"\nApplied at EUR {PRICE_SCREEN:.2f}: "
      f"{int(panel['_price_screened'].sum()):,} observations set to missing")
print(f"\nMonthly excess returns: {panel['retx'].notna().sum():,}")
print(panel["retx"].describe(percentiles=[.01, .5, .99]).round(4).to_string())

/tmp/ipykernel_2155/2560170962.py:35: FutureWarning: The default fill_method='ffill' in SeriesGroupBy.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  p["ret"] = ri.groupby(level="symbol").pct_change()
/tmp/ipykernel_2155/2560170962.py:47: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  bad_next = bad.groupby(level="symbol").shift(-1).fillna(False).astype(bool)


Panel: 870,254 security-months, 6,156 securities
Removed by reversal filter: 245

Effect of the price screen on excess returns:
  threshold EUR 0.00: n=  724,584  std=  11.800  max=  10000.0
  threshold EUR 0.10: n=  708,232  std=   0.278  max=     74.1
  threshold EUR 0.25: n=  695,384  std=   0.237  max=     60.4
  threshold EUR 1.00: n=  642,644  std=   0.206  max=     60.4

Applied at EUR 0.25: 29,675 observations set to missing

Monthly excess returns: 695,384
count    695384.0000
mean          0.0072
std           0.2373
min          -1.0014
1%           -0.3799
50%          -0.0018
99%           0.5352
max          60.4290


## 4. Market benchmark

The benchmark is not specified in Drobetz and Otto (2021). The earlier study draws
its universe from the STOXX Europe 600 and states that this index serves as its
value-weighted market benchmark, indicating that a broad capitalisation-weighted
European index is intended.

A euro-area composite is therefore constructed from the four national total market
indices, weighted by the aggregate market capitalisation of the sample securities in
each country. Weights are taken at the end of the preceding month, so that no
contemporaneous information enters them. The aggregate European index is retained
for a robustness check, but is not used as the primary benchmark: it covers the
European Union rather than the euro area, and so includes markets outside the
currency union.

In [6]:
INDEX_CODES = {"DE": "TOTMKBD", "FR": "TOTMKFR", "IT": "TOTMKIT", "ES": "TOTMKES"}

mkt, weights = build_composite_index(idx_raw, panel, master, INDEX_CODES)

print(f"Composite benchmark: {mkt.notna().sum():,} weekly returns")
print(f"  {mkt.index.min():%Y-%m} to {mkt.index.max():%Y-%m}")
print(f"  mean {mkt.mean()*100:.3f}% per week, std {mkt.std()*100:.2f}%")
print("\nAverage country weights:")
print((weights.mean() * 100).round(1).astype(str).add("%").to_string())

Composite benchmark: 1,561 weekly returns
  1996-01 to 2025-12
  mean 0.205% per week, std 2.76%

Average country weights:
country
TOTMKBD    28.6%
TOTMKES    12.0%
TOTMKFR    37.5%
TOTMKIT    21.9%


## 5. Characteristics

### 5.1 Market-based

| # | Characteristic | Definition (Drobetz and Otto, 2021) |
|---|---|---|
| 1 | `size` | log market capitalisation of equity at the end of the prior month |
| 5 | `ret_2_12` | excess return from month $-12$ to month $-2$ |
| 10 | `ret_1` | excess return of the prior month |
| 11 | `ret_12_36` | excess return from month $-36$ to month $-12$ |
| 6 | `issues_1_36` | log growth in split-adjusted shares outstanding, months $-36$ to $-1$ |
| 20 | `issues_1_12` | log growth in split-adjusted shares outstanding, months $-12$ to $-1$ |
| 15 | `turnover` | average monthly turnover volume, months $-12$ to $-1$ |
| 12 | `dy` | dividends per share over the prior 12 months, divided by price |
| 21 | `fcdispersion` | log standard deviation of I/B/E/S EPS forecasts minus log average absolute forecast |

**Split-adjusted shares.** Genuine issuance must be separated from share counts that
change mechanically at splits. In these data the identity
$\mathrm{MV} = (\mathrm{P}/\mathrm{AF}) \times \mathrm{NOSH}/1000$ holds within 5%
for 94.5% of observations, against 79.5% without the adjustment factor, so
`NOSH / AF` is the split-adjusted count.

**Turnover.** The source describes it as "average monthly turnover volume". Raw
volume would be of the order of millions of shares, whereas the reported mean is
0.02; scaling by shares outstanding is therefore implied. The earlier
large-capitalisation sample reports 0.13 against 0.02 for the broader later one,
which is consistent with a ratio rather than a level.

**Dividend yield.** The Datastream dividend per share is already a trailing
twelve-month figure reported monthly, not a monthly dividend. Verified against the
`DY` datatype: `DPS/P` reproduces it exactly, whereas cumulating `DPS` over twelve
months overstates the yield by a factor of about twelve.

In [7]:
panel = add_market_characteristics(panel)
panel = add_target(panel)

MARKET = ["size", "ret_1", "ret_2_12", "ret_12_36",
          "issues_1_12", "issues_1_36", "turnover", "dy", "fcdispersion"]
for ch in MARKET:
    print(f"  {ch:14s} {panel[ch].notna().sum():>10,}  ({100*panel[ch].notna().mean():>5.1f}%)")

  size              728,296  ( 83.7%)
  ret_1             695,384  ( 79.9%)
  ret_2_12          631,462  ( 72.6%)
  ret_12_36         520,887  ( 59.9%)
  issues_1_12       688,367  ( 79.1%)
  issues_1_36       570,296  ( 65.5%)
  turnover          649,281  ( 74.6%)
  dy                730,993  ( 84.0%)
  fcdispersion      389,937  ( 44.8%)


### 5.2 Beta and volatility

| # | Characteristic | Definition |
|---|---|---|
| 13 | `beta` | market beta, estimated from weekly returns from month $-36$ to month $-1$ |
| 14 | `vola` | standard deviation, estimated from weekly returns from month $-12$ to month $-1$ |

Estimated by rolling covariance rather than repeated regressions. A minimum of two
thirds of the window is required, so that a security with a short history does not
receive an estimate from a handful of observations.

In [8]:
bv = beta_and_vola(weekly, mkt, beta_weeks=156, vola_weeks=52,
                   min_beta=104, min_vola=35)
panel = panel.join(bv, how="left")

print(f"  beta  {panel['beta'].notna().sum():>10,}  ({100*panel['beta'].notna().mean():>5.1f}%)")
print(f"  vola  {panel['vola'].notna().sum():>10,}  ({100*panel['vola'].notna().mean():>5.1f}%)")
print()
print(panel[["beta", "vola"]].describe(percentiles=[.01, .5, .99]).round(3).to_string())

/tmp/ipykernel_2155/633203085.py:50: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  r = w.pct_change()


  beta     608,647  ( 69.9%)
  vola     689,406  ( 79.2%)

             beta        vola
count  608647.000  689406.000
mean        0.571       0.116
std         5.471       6.202
min      -227.808       0.000
1%         -0.671       0.000
50%         0.479       0.051
99%         2.084       0.427
max      1933.808    1386.751


**Units of volatility.** Drobetz and Otto report a mean of 0.32. A weekly return
standard deviation of that magnitude would be implausible for a typical equity; the
figure is consistent with an annualised measure, a weekly standard deviation of
roughly 4.4% scaled by $\sqrt{52}$. The choice affects only the scale of the
variable and not its cross-sectional ranking, which is what enters the models after
the rank transformation, but the annualised form is adopted for comparability.

In [9]:
ANNUALISE_VOLA = True

if ANNUALISE_VOLA:
    panel["vola"] = panel["vola"] * np.sqrt(52)
    print(f"Annualised. Mean {panel['vola'].mean():.3f}, "
          f"median {panel['vola'].median():.3f} (Drobetz and Otto report 0.32)")
else:
    print(f"Weekly units. Mean {panel['vola'].mean():.4f}")

Annualised. Mean 0.839, median 0.369 (Drobetz and Otto report 0.32)


### 5.3 Accounting-based

| # | Characteristic | Definition |
|---|---|---|
| 2 | `bm` | log book value of equity minus log market capitalisation |
| 3 | `operatingprofitability` | operating profit divided by average total assets |
| 4 | `totalassetgrowth` | log total asset growth in the prior fiscal year |
| 7 | `accrualschange` | working capital accruals (Sloan, 1996) |
| 8 | `roa` | income before extraordinary items divided by average total assets |
| 9 | `investment` | capital expenditures divided by net sales and revenues |
| 16 | `debttoprice` | short-term plus long-term debt divided by market capitalisation |
| 17 | `salestoprice` | net sales and revenues divided by market capitalisation |
| 18 | `cftoprice` | net income plus depreciation, depletion and amortisation, over market capitalisation |
| 19 | `earningstoprice` | net income divided by market capitalisation |
| 22 | `grossprofitability` | net sales and revenues minus cost of goods sold, over average total assets |

**Reporting lag.** Each fiscal year becomes usable four months after its *reported*
period end (item WC05350), not after an assumed December year-end. This matters: a
fifth of firm-year observations in the sample close their accounts in a month other
than December. The merge onto the monthly panel is an as-of join, so each month
carries the most recent fiscal year that was already available.

**Units.** Worldscope reports in thousands of currency units and Datastream market
value in millions. Verified against the sample: the ratio of common equity to market
value has a median of 339, implying a book-to-market of 0.34, and total assets are
1.33 times market value at the median. Accounting items are therefore divided by one
thousand.

In [10]:
acc = accounting_available(annual, fye, lag_months=4)

lag_check = ((acc["available"].dt.year - acc["fye"].dt.year) * 12 +
             (acc["available"].dt.month - acc["fye"].dt.month))
print("Reporting lag in months:", lag_check.value_counts().to_dict())

nondec = (acc["fye"].dt.month != 12).mean()
print(f"Firm-years with a non-December fiscal year end: {100*nondec:.1f}%")
print(f"\nAccounting observations: {len(acc):,} | securities: {acc['symbol'].nunique():,}")

Reporting lag in months: {4: 76620}
Firm-years with a non-December fiscal year end: 11.2%

Accounting observations: 76,620 | securities: 5,200


In [11]:
panel = add_accounting_characteristics(panel, acc)

ACCOUNTING = ["bm", "operatingprofitability", "totalassetgrowth", "accrualschange",
              "roa", "investment", "debttoprice", "salestoprice", "cftoprice",
              "earningstoprice", "grossprofitability"]
for ch in ACCOUNTING:
    print(f"  {ch:24s} {panel[ch].notna().sum():>10,}  ({100*panel[ch].notna().mean():>5.1f}%)")

  bm                          621,952  ( 71.5%)
  operatingprofitability      711,548  ( 81.8%)
  totalassetgrowth            739,969  ( 85.0%)
  accrualschange              355,936  ( 40.9%)
  roa                         739,839  ( 85.0%)
  investment                  705,340  ( 81.0%)
  debttoprice                 646,067  ( 74.2%)
  salestoprice                652,341  ( 75.0%)
  cftoprice                   621,127  ( 71.4%)
  earningstoprice             651,071  ( 74.8%)
  grossprofitability          615,605  ( 70.7%)


## 6. Comparison with the source study

Levels are not expected to coincide. Drobetz and Otto retain only firms with
complete information across all twenty-two characteristics, and require at least
fifty firms above €25 million in each month; the sample here is the full universe
before any such restriction, and is dominated by small and delisted securities.

In [12]:
CHARS = MARKET + ["beta", "vola"] + ACCOUNTING

reference = {
    "size": 13.22, "bm": -0.55, "operatingprofitability": 0.06,
    "totalassetgrowth": 0.07, "ret_2_12": 0.09, "issues_1_36": 0.09,
    "accrualschange": -0.04, "roa": 0.05, "investment": 1.02, "ret_1": 0.01,
    "ret_12_36": 0.09, "dy": 0.03, "beta": 0.83, "vola": 0.32, "turnover": 0.02,
    "debttoprice": 0.84, "salestoprice": 2.16, "cftoprice": 0.14,
    "earningstoprice": 0.04, "issues_1_12": 0.02, "fcdispersion": -2.05,
    "grossprofitability": 0.30,
}

cmp = pd.DataFrame({
    "mean": panel[CHARS].mean(),
    "median": panel[CHARS].median(),
    "std": panel[CHARS].std(),
    "coverage%": (100 * panel[CHARS].notna().mean()).round(1),
    "DO_2021_mean": pd.Series(reference),
})
print(cmp.round(3).to_string())

                          mean  median       std  coverage%  DO_2021_mean
accrualschange          -0.041  -0.041     0.243       40.9         -0.04
beta                     0.571   0.479     5.471       69.9          0.83
bm                      -0.436  -0.467     1.366       71.5         -0.55
cftoprice               -0.831   0.084   118.361       71.4          0.14
debttoprice             14.440   0.321   378.874       74.2          0.84
dy                       0.226   0.004    32.867       84.0          0.03
earningstoprice         -2.378   0.037   133.666       74.8          0.04
fcdispersion            -1.977  -2.121     1.001       44.8         -2.05
grossprofitability       0.258   0.207     0.430       70.7          0.30
investment               0.330   0.036    10.341       81.0          1.02
issues_1_12              0.038   0.000     0.270       79.1          0.02
issues_1_36              0.118   0.000     0.473       65.5          0.09
operatingprofitability   0.022   0.047

**Two figures require comment.**

`size` is reported in logs and therefore depends on the unit in which market value is
expressed. Drobetz and Otto report 13.22, which corresponds to €551 billion if market
value is in millions and €551 million if in thousands; only the latter is plausible
for a sample of European firms, so their market value is in thousands against
millions here, a constant difference of $\log 1000 = 6.9$. The transformation applied
before estimation is a cross-sectional rank, which is invariant to the unit.

`investment` is defined as capital expenditures over net sales, for which the
reported mean of 1.02 would imply capital expenditure equal to turnover. The value
obtained here is an order of magnitude lower and economically more plausible, which
suggests the reported figure reflects a different normalisation.

## 7. Observations per month

The number of securities with a complete set of characteristics determines the size
of each cross-section, and is the point of comparison with the 832 firms per month
reported by Drobetz and Otto.

In [13]:
complete = panel[CHARS + ["target"]].notna().all(axis=1)
per_month_all = panel.dropna(subset=["target"]).groupby(level="date").size()
per_month_cc = panel[complete].groupby(level="date").size()

print(f"Security-months with a target        : {panel['target'].notna().sum():>10,}")
print(f"Security-months complete on all 22   : {int(complete.sum()):>10,}")
print(f"Securities appearing in complete case: "
      f"{panel[complete].index.get_level_values('symbol').nunique():>10,}")
print("\nSecurities per month, selected years:")
print(f"{'year':>6} {'with target':>12} {'complete case':>15}")
for y in [2000, 2005, 2010, 2015, 2020, 2025]:
    a = per_month_all[per_month_all.index.year == y]
    b = per_month_cc[per_month_cc.index.year == y]
    print(f"{y:>6} {a.mean() if len(a) else 0:>12.0f} {b.mean() if len(b) else 0:>15.0f}")
print("\nDrobetz and Otto (2021) report 832 firms per month.")

Security-months with a target        :    695,384
Security-months complete on all 22   :    139,040
Securities appearing in complete case:      1,428

Securities per month, selected years:
  year  with target   complete case
  2000         2203             282
  2005         1979             349
  2010         2048             516
  2015         1802             429
  2020         1779             503
  2025         1651             525

Drobetz and Otto (2021) report 832 firms per month.


## 8. Save

In [14]:
keep = (["ret", "rf", "retx", "target", "MV", "P", "NOSH", "AF", "VO", "DPS"]
        + CHARS + ["fy", "fye", "available"])
keep = [c for c in keep if c in panel.columns]

out_df = panel[keep].reset_index()
out_df.to_parquet(os.path.join(DATA_DIR, "panel_characteristics.parquet"), index=False)

p = os.path.join(DATA_DIR, "panel_characteristics.parquet")
print(f"  panel_characteristics.parquet  {len(out_df):>12,} rows x {out_df.shape[1]} cols  "
      f"({os.path.getsize(p)/1e6:.1f} MB)")
print("\nNext: sample filtering, winsorisation at 1% and 99%, and the")
print("cross-sectional rank transformation onto the interval (-1, +1).")

  panel_characteristics.parquet       870,254 rows x 37 cols  (97.9 MB)

Next: sample filtering, winsorisation at 1% and 99%, and the
cross-sectional rank transformation onto the interval (-1, +1).


In [15]:
print("beta  :", round(panel.loc[complete, "beta"].median(), 3), " (D&O 0.83)")
print("dy    :", round(panel.loc[complete, "dy"].median(), 4), " (D&O 0.03)")
print("size  :", round(panel.loc[complete, "size"].median(), 2), " (D&O 13.22 in thousand)")

beta  : 0.849  (D&O 0.83)
dy    : 0.0183  (D&O 0.03)
size  : 6.67  (D&O 13.22 in migliaia)
